# 《计算机视觉》作业 3 — 基于 Oxford-IIIT Pet Dataset 的图像分割实验

**传统图像处理方法 + 经典深度学习分割网络 FCN**

本 notebook 在 macOS 与 Windows 上都可以运行,推荐使用 `conda env create -f ../environment.yml` 创建环境,
随后在 VS Code 中右上角 "Select Kernel" 选择 `cv-pet-seg` 即可执行。

## 内容大纲

1. 环境准备 & 工具函数导入
2. 数据集加载 (torchvision `OxfordIIITPet`, trimap → 二值 mask)
3. 数据预览
4. 传统方法 — Otsu / Canny+形态学 / MeanShift
5. 深度学习方法 — FCN-ResNet50 预训练推理
6. 评价指标:IoU / Dice / Accuracy / Precision / Recall / F1
7. 成功 / 失败样本可视化
8. (可选) 在小批数据上对 FCN 进行轻量微调
9. 结论与改进方向

## 1. 环境准备

**首次运行**前请确保已经创建 conda 环境:
```bash
conda env create -f ../environment.yml
conda activate cv-pet-seg
```
VS Code 中:左下角解释器 / Notebook 右上角 Kernel → 选择 **cv-pet-seg**。

下面这一格把项目根目录加入 `sys.path`,这样就可以直接 `import src.*` 了。

In [ ]:
import os, sys, time, pathlib
import numpy as np
import matplotlib.pyplot as plt
import torch

PROJECT_ROOT = pathlib.Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import data_utils, traditional, dl_models, metrics, visualize

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("project root :", PROJECT_ROOT)
print("torch        :", torch.__version__)
print("device       :", DEVICE)
print("numpy        :", np.__version__)

## 2. 加载 Oxford-IIIT Pet Dataset

* `target_types="segmentation"` 返回 trimap (PIL.Image)
* trimap 中:`1` = 宠物前景,`2` = 背景,`3` = 边界 → 我们将 `{1,3}` 合并为前景 (255),`2` → 0
* 第一次运行会从官方源下载约 800MB,需保持网络畅通;如下载缓慢,可参考作业提供的网盘链接 (见 README) 把 `images.tar.gz` 与 `annotations.tar.gz` 放入 `../data/oxford-iiit-pet/` 后再运行

In [ ]:
DATA_ROOT = str(PROJECT_ROOT / "data")
IMAGE_SIZE = 128

train_ds = data_utils.load_oxford_pet_dataset(root=DATA_ROOT, split="trainval", download=True)
test_ds  = data_utils.load_oxford_pet_dataset(root=DATA_ROOT, split="test",     download=True)
print("trainval size:", len(train_ds))
print("test     size:", len(test_ds))

In [ ]:
N_EVAL = 60
test_samples  = data_utils.build_sample_pool(test_ds,  image_size=IMAGE_SIZE, max_samples=N_EVAL, seed=2026)
train_samples = data_utils.build_sample_pool(train_ds, image_size=IMAGE_SIZE, max_samples=20,     seed=1)
print(f"test samples used : {len(test_samples)}")
print(f"train samples for demo : {len(train_samples)}")

## 3. 数据预览

随机展示几张训练图与其对应的二值 mask,以确认 trimap → binary mask 的转换正确。

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for col in range(4):
    s = train_samples[col]
    axes[0, col].imshow(s.image_rgb)
    axes[0, col].set_title(s.filename, fontsize=8)
    axes[0, col].axis("off")
    axes[1, col].imshow(s.gt_mask, cmap="gray", vmin=0, vmax=255)
    axes[1, col].set_title("binary mask")
    axes[1, col].axis("off")
plt.tight_layout(); plt.show()

## 4. 传统方法

**任选其二**(本 notebook 默认跑全部三种,便于对比):
- Otsu (Lab 色彩空间 a* 通道 + 灰度通道融合)
- Canny 边缘 + 形态学闭运算 + 孔洞填充
- MeanShift 区域分割

三个函数都在 `src/traditional.py` 中实现,输出都是 HxW uint8 二值 mask。

In [ ]:
preds_traditional = {"Otsu": [], "Canny+Morph": [], "MeanShift": []}
t0 = time.time()
for s in test_samples:
    for m in preds_traditional:
        preds_traditional[m].append(traditional.run_traditional(m, s.image_rgb))
print(f"traditional methods done in {time.time()-t0:.1f}s, n={len(test_samples)}")

## 5. 深度学习方法

直接加载 torchvision 的 **FCN-ResNet50** (在 Pascal VOC 上训练,COCO 子集预训练) 权重做推理。
将 VOC 类别中的 `cat (8)` 与 `dog (12)` 合并为宠物前景,其余视为背景。

> 备注:首次运行会下载 ~140MB 的权重文件。

In [ ]:
fcn = dl_models.load_fcn(device=DEVICE)
batch_tensor, gts = data_utils.build_tensor_batch(test_samples)
print("input batch:", tuple(batch_tensor.shape))

t0 = time.time()
pred_fcn = dl_models.predict_masks(fcn, batch_tensor, device=DEVICE, target_size=IMAGE_SIZE)
print(f"FCN inference done in {time.time()-t0:.1f}s")

## 6. 评价:IoU / Dice / Accuracy / Precision / Recall / F1

对四种方法 (3 个传统 + 1 个 FCN) 在同一批测试图像上计算指标,并以 Markdown 表格输出,便于直接复制到报告中。

In [ ]:
preds_by_method = dict(preds_traditional)
preds_by_method["FCN-ResNet50"] = pred_fcn

metrics_by_method = {}
for name, preds in preds_by_method.items():
    metrics_by_method[name] = [
        metrics.compute_all(p, s.gt_mask, s.filename)
        for p, s in zip(preds, test_samples)
    ]

table_md = metrics.metrics_to_table(metrics_by_method)
print(table_md)

In [ ]:
names = list(metrics_by_method.keys())
mean_iou  = [metrics.aggregate(metrics_by_method[n])["mean_iou"]  for n in names]
mean_dice = [metrics.aggregate(metrics_by_method[n])["mean_dice"] for n in names]

x = np.arange(len(names)); w = 0.35
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - w/2, mean_iou,  width=w, label="mean IoU",  color="#4C72B0")
ax.bar(x + w/2, mean_dice, width=w, label="mean Dice", color="#DD8452")
for i, v in enumerate(mean_iou):  ax.text(i - w/2, v + 0.01, f"{v:.2f}", ha="center", fontsize=9)
for i, v in enumerate(mean_dice): ax.text(i + w/2, v + 0.01, f"{v:.2f}", ha="center", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(names, rotation=10)
ax.set_ylim(0, 1.05); ax.set_ylabel("score")
ax.set_title(f"Pet segmentation — metrics on {len(test_samples)} test images")
ax.legend(); plt.tight_layout(); plt.show()

## 7. 成功 / 失败样本可视化

为每种方法挑出 IoU 最高的 3 个 (成功) 和 IoU 最低的 3 个 (失败) 样本,
展示「原图 | 真值 | 预测 | 误差图」。

误差图颜色含义:**绿色=TP, 红色=FP (多预测), 蓝色=FN (漏预测), 黑色=TN**。

In [ ]:
def top_bottom_indices(metric_list, k=3):
    ious = np.array([m.iou for m in metric_list])
    order = np.argsort(ious)
    bottom = order[:k].tolist()
    top    = order[-k:][::-1].tolist()
    return top, bottom

for method, ms in metrics_by_method.items():
    top, bottom = top_bottom_indices(ms, k=3)
    print(f"\n===== {method} =====")
    print(" success (top  IoU):", [(i, round(ms[i].iou, 3)) for i in top])
    print(" failure (bot  IoU):", [(i, round(ms[i].iou, 3)) for i in bottom])
    visualize.show_gallery(test_samples, {method: preds_by_method[method]}, top,    section_title=f"{method} success")
    visualize.show_gallery(test_samples, {method: preds_by_method[method]}, bottom, section_title=f"{method} failure")

## 8. (可选) FCN 轻量微调

torchvision 的 FCN 预训练权重已经能识别 cat/dog 大类,但对 Oxford-IIIT Pet 的某些精细品种 (例如毛色与背景近似的样本) 仍会出错。
下面演示在 100 张训练图上对 **classifier 头** 做 3 个 epoch 的轻量微调,仅作为示范,可根据自身硬件调整规模。

> 如果只需要完成作业的最小要求 (推理 + 评价),可以跳过本节。

In [ ]:
DO_FINETUNE = False  # 设为 True 启用

if DO_FINETUNE:
    finetune_pool = data_utils.build_sample_pool(train_ds, image_size=IMAGE_SIZE, max_samples=100, seed=7)
    ft_tensor, ft_gts = data_utils.build_tensor_batch(finetune_pool)
    ft_target = torch.from_numpy(np.stack([(g > 0).astype(np.int64) for g in ft_gts])).to(DEVICE)

    model = dl_models.load_fcn(device=DEVICE)
    for p in model.backbone.parameters(): p.requires_grad = False
    optim = torch.optim.AdamW(model.classifier.parameters(), lr=1e-4)
    loss_fn = torch.nn.CrossEntropyLoss()
    bs = 8
    model.train()
    for epoch in range(3):
        perm = torch.randperm(ft_tensor.size(0))
        for i in range(0, ft_tensor.size(0), bs):
            idx = perm[i:i+bs]
            x = ft_tensor[idx].to(DEVICE)
            y = ft_target[idx]
            logits = model(x)["out"]  # 21 类
            cat_dog_logit = torch.stack([
                logits[:, [0,1,2,3,4,5,6,7,9,10,11,13,14,15,16,17,18,19,20]].max(dim=1).values,
                torch.maximum(logits[:, 8], logits[:, 12])
            ], dim=1)
            loss = loss_fn(cat_dog_logit, y)
            optim.zero_grad(); loss.backward(); optim.step()
        print(f"epoch {epoch+1} loss={loss.item():.4f}")
    print("finetune done")

## 9. 结论与改进方向

根据上方的指标表与可视化结果,可以总结出:

1. **传统方法**
    - *Otsu* 在前景背景对比明显的样本上表现较好,但在背景颜色与宠物相近 (如黑猫卧黑沙发) 时容易整张图被判为前景。
    - *Canny+形态学* 对纹理/边缘明显的样本有效,但毛发边缘往往被打散,孔洞填充后 mask 也容易出现「方形包络」。
    - *MeanShift* 受参数 `sp`, `sr` 影响较大,运行时间也最长,精度提升相对有限。
2. **深度学习方法 (FCN-ResNet50)** 在 IoU/Dice 上显著优于传统方法,对复杂背景具有更强的鲁棒性;但对小目标 / 大幅截断 / 罕见姿态仍可能漏检。
3. **改进方向**
    - 引入 DeepLabV3+ 或 Segformer 等更强 backbone
    - 使用 Oxford-IIIT Pet 的训练子集做完整微调而非仅头部微调
    - 数据增强:随机裁剪 / 颜色抖动 / CutMix
    - 结合传统先验 (如 GrabCut) 做后处理,改善边界细节

> 详细分析见 `../report/REPORT_TEMPLATE.md`,小组分工见 `../report/CONTRIBUTIONS.md`。